# CBCB — Exploration & Pipeline Walkthrough

Content-Based Captivation Behavior — reproduction of *Predicting User Behavior on Video Streaming by Using Watch-Time Duration Analysis* (Knowledge-Based Systems, 2025).

This notebook walks through the full pipeline: **generate → preprocess → label (CBCB-S/R) → features → train → evaluate**, with EDA visualizations along the way.

> Run from the project root, or adjust `sys.path` as below.

In [ ]:
import sys, os
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, dataset_generator, cbcb_s, cbcb_r, feature_engineering, train, evaluate
from src import visualization as viz
from src.preprocessing import preprocess_pipeline, correlation_matrix
print('Project root:', ROOT)

## 1. Generate a synthetic dataset

Same schema as the STC/JAWWY data, with built-in sequential/revert structure and injected outliers.

In [ ]:
df = dataset_generator.generate_dataset(n_rows=20_000, n_users=800)
df.head()

In [ ]:
df.describe(include='all').T

## 2. Exploratory Data Analysis

In [ ]:
viz.watch_time_distribution(df);
viz.genre_distribution(df);
viz.user_activity(df);

## 3. Preprocess: clean → IQR → one-hot → min-max

`X' = fs(fe(fc(X)))`

In [ ]:
processed, transformers, outlier_stats = preprocess_pipeline(df)
print('Removed outliers:', outlier_stats['n_removed'])
viz.outliers_before_after(outlier_stats);
processed.head()

In [ ]:
viz.correlation_heatmap(correlation_matrix(processed));

## 4. Generate CBCB-S and CBCB-R labels

In [ ]:
s_labelled = cbcb_s.generate_labels(processed)
r_labelled = cbcb_r.generate_labels(processed)
print('CBCB-S label distribution:', s_labelled[config.LABEL_COL_S].value_counts().to_dict())
print('CBCB-R label distribution:', r_labelled[config.LABEL_COL_R].value_counts().to_dict())

## 5. Build features and train models (CBCB-S)

In [ ]:
X, y, names = feature_engineering.build_feature_matrix(s_labelled, config.LABEL_COL_S)
bundle = train.train_all(X, y, 'cbcb_s', names, tune=False, include_boosting=False, persist=False)
evaluate.comparison_table(bundle['results'])

In [ ]:
for metric in config.METRIC_KEYS:
    viz.metric_bar(bundle['results'], metric);
print('Best per metric:', bundle['best'])
print('Recommended:', evaluate.recommend_model(bundle['results']))

## 6. CBCB-R training & confusion matrices

In [ ]:
Xr, yr, names_r = feature_engineering.build_feature_matrix(r_labelled, config.LABEL_COL_R)
bundle_r = train.train_all(Xr, yr, 'cbcb_r', names_r, tune=False, include_boosting=False, persist=False)
for key, m in bundle_r['results'].items():
    viz.confusion_heatmap(m['confusion_matrix'], m['classes'], title=m['name']);
evaluate.comparison_table(bundle_r['results'])

## 7. Takeaways

- The **Decision Tree** is typically the strongest conventional model (matches the paper).
- **CBCB-R** generally outperforms **CBCB-S** thanks to the richer revert signal.
- Watch-time (scaled) carries meaningful feature importance, supporting the paper's hypothesis.

For the full tuned run across both tasks, use the CLI: `python main.py --all`.